# dimensions


In [ ]:
"""
Popula las 8 dimensiones del modelo dimensional (esquema dw) desde los datos
de staging (esquema stage). El contrato de columnas/PK es scripts/dw/ddl_oracle.sql
— los nombres y el orden de columnas de salida coinciden con ese DDL.

Limpieza Kimball: los NULL en dimensiones se reemplazan con casos genéricos
  - Texto: "No aplica" / "Desconocido"
  - Numérico: -1
  - Fecha sin mes/dia: mes_num = 0
Cada dimensión incluye una fila genérica (id=0 o id=-1) para registros huérfanos
en los hechos — las FKs nunca quedan NULL (§integridad referencial).

dim_ine_perfil: se canonicalizan las categorías de INE para que no se fragmenten
  ('Casado'/'Casado(a)', 'Básica'/'Básico', 'Ninguna'/'Ninguno' = mismo valor).
dim_causa: clave natural = cie-10_code (si válido) ELSE gbd_code. Se cargan tanto
  las causas CIE-10 (INE/Panamá) como las GBD (IHME/Panamá) y las 4 super-categorías
  de WHO (icd10_group) — así los hechos enganchan al grano más fino disponible.

Orden de ejecución: dimensions.py → facts.py
"""

from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import (
    IntegerType, StringType, BooleanType, TimestampType,
)
from pyspark.sql.window import Window

SCHEMA_DW = "semi2.dm_mortality"

# IDs genéricos para filas "No aplica" / "Desconocido"
GEN_ID_NEG  = -1   # surrogate genérico para la mayoría (causa, geo, perfil, etc.)
GEN_ID_ZERO = 0    # genero desconocido, mes base anual

# Formato CIE-10 individual válido (A00, R98X, …). Los rangos ("C82-C85") no.
CIE10_RE = r"^[A-Z][0-9]{2}[0-9A-Z]?$"

# WHO Mortality "deaths by age group" no trae CIE-10/GBD por causa, solo el
# super-grupo icd10_group (4 categorías ~ GBD Nivel 1). Se cargan como causas
# con un código sintético para que fact_who_deaths tenga causa real (no -1).
WHO_ICD10_CAT = {
    "communicable_maternal_perinatal_and_nutritional_conditions":
        ("WHO_I",   "Communicable, maternal, perinatal and nutritional conditions"),
    "noncommunicable_diseases":
        ("WHO_II",  "Noncommunicable diseases"),
    "injuries":
        ("WHO_III", "Injuries"),
    "ill_defined_diseases":
        ("WHO_IV",  "Ill-defined diseases"),
}


def _save(df: DataFrame, table: str) -> None:
    full = f"{SCHEMA_DW}.{table}"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full)
    print(f"[DW] {full} — {df.count()} filas")


def _assign_id(df: DataFrame, id_col: str, *order_cols: str) -> DataFrame:
    return df.withColumn(
        id_col,
        F.row_number().over(Window.orderBy(*order_cols)).cast(IntegerType())
    )


# ═══════════════════════════════════════════════════════════════════════
# DIM_ETARIO (hardcoded, grid canónico §5.2)  — DDL: grupo_edad (no _nombre)
# ═══════════════════════════════════════════════════════════════════════

_DIM_ETARIO = [
    (1,  "LT1",   "Menores de 1",      "Niñez",           0,   0),
    (2,  "01-04", "1 a 4",             "Niñez",           1,   4),
    (3,  "05-09", "5 a 9",             "Niñez",           5,   9),
    (4,  "10-14", "10 a 14",           "Niñez",          10,  14),
    (5,  "15-19", "15 a 19",           "Juventud",       15,  19),
    (6,  "20-24", "20 a 24",           "Juventud",       20,  24),
    (7,  "25-29", "25 a 29",           "Juventud",       25,  29),
    (8,  "30-34", "30 a 34",           "Adultez",        30,  34),
    (9,  "35-39", "35 a 39",           "Adultez",        35,  39),
    (10, "40-44", "40 a 44",           "Adultez",        40,  44),
    (11, "45-49", "45 a 49",           "Adultez",        45,  49),
    (12, "50-54", "50 a 54",           "Adultez",        50,  54),
    (13, "55-59", "55 a 59",           "Adultez",        55,  59),
    (14, "60-64", "60 a 64",           "Adultez",        60,  64),
    (15, "65-69", "65 a 69",           "Vejez",          65,  69),
    (16, "70-74", "70 a 74",           "Vejez",          70,  74),
    (17, "75-79", "75 a 79",           "Vejez",          75,  79),
    (18, "80-84", "80 a 84",           "Vejez",          80,  84),
    (19, "85+",   "85 y más",          "Vejez",          85, -1),
    (98, "UNK",   "No especificada",   "No especificado", -1, -1),
    (99, "ALL",   "Todas las edades",  "Total",           0, -1),
]


def build_dim_etario() -> DataFrame:
    return spark.createDataFrame(
        _DIM_ETARIO,
        ["id_etario", "grupo_edad_codigo", "grupo_edad",
         "categoria_etaria", "edad_minima", "edad_maxima"],
    ).select(
        F.col("id_etario").cast(IntegerType()),
        "grupo_edad_codigo", "grupo_edad",
        "categoria_etaria",
        F.col("edad_minima").cast(IntegerType()),
        F.col("edad_maxima").cast(IntegerType()),
    )


# ═══════════════════════════════════════════════════════════════════════
# DIM_SOURCE (hardcoded)  — DDL: id_source, source, ingestion_ts, record_hash
# ═══════════════════════════════════════════════════════════════════════

_DIM_SOURCE = [
    (1, "INE Guatemala"),
    (2, "IHME GBD 2023"),
    (3, "MSPAS Guatemala"),
    (4, "INEC Panamá"),
    (5, "WHO Mortality"),
]


def build_dim_source() -> DataFrame:
    return (
        spark.createDataFrame(_DIM_SOURCE, ["id_source", "source"])
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("record_hash", F.sha2(F.col("source"), 256))
        .select("id_source", "source", "ingestion_ts", "record_hash")
    )


# ═══════════════════════════════════════════════════════════════════════
# DIM_GENERO (hardcoded + genérico id=0)
# ═══════════════════════════════════════════════════════════════════════

_GENERO_CANON = [
    (0, "Desconocido"),   # fila genérica para sexo no mapeado
    (1, "Hombre"),
    (2, "Mujer"),
    (3, "Ambos"),
]


def build_dim_genero() -> DataFrame:
    return spark.createDataFrame(
        _GENERO_CANON,
        ["id_genero", "sexo_nombre"],
    ).withColumn("sexo_codigo", F.col("sexo_nombre")) \
     .select("id_genero", "sexo_codigo", "sexo_nombre")


# ═══════════════════════════════════════════════════════════════════════
# DIM_CAUSA  — DDL: id_causa, "cie-10_code", "cie-10_nombre", gbd_code, gbd_nombre
# Clave natural causa_nk = cie-10_code (si formato válido) ELSE gbd_code.
# ═══════════════════════════════════════════════════════════════════════

def _valid_cie10(col):
    return F.when(
        col.isNotNull() & F.upper(F.trim(col)).rlike(CIE10_RE),
        F.upper(F.trim(col))
    ).otherwise(F.lit(None).cast(StringType()))


def build_dim_causa() -> DataFrame:
    dfs = []

    # --- IHME: solo GBD ---
    if spark.catalog.tableExists("stage.ihme"):
        dfs.append(
            spark.read.table("stage.ihme").select(
                F.lit(None).cast(StringType()).alias("cie10_code"),
                F.lit(None).cast(StringType()).alias("cie10_nombre"),
                F.col("gbd_code"), F.col("gbd_nombre"),
            )
        )

    # --- INE: CIE-10 (siempre poblado) + GBD (parcial) ---
    if spark.catalog.tableExists("stage.ine"):
        dfs.append(
            spark.read.table("stage.ine").select(
                _valid_cie10(F.col("cie10_code")).alias("cie10_code"),
                F.col("cie10_nombre"),
                F.col("gbd_code"), F.col("gbd_nombre"),
            )
        )

    # --- Panamá: CIE-10 (solo individuales válidos) + GBD + causa_codigo fallback
    if spark.catalog.tableExists("stage.panama"):
        panama = spark.read.table("stage.panama")
        gbd = F.when(F.col("gbd_code") == "null", F.lit(None).cast(StringType())) \
               .otherwise(F.col("gbd_code"))
        gbd_fb = F.coalesce(gbd, F.col("causa_codigo").cast(StringType()))
        gbd_nombre_fb = F.when(gbd.isNotNull(), F.col("gbd_nombre")) \
                          .otherwise(F.col("causa"))
        dfs.append(
            panama.select(
                _valid_cie10(F.col("cie10_code")).alias("cie10_code"),
                F.col("cie10_nombre"),
                gbd_fb.alias("gbd_code"), gbd_nombre_fb.alias("gbd_nombre"),
            )
        )

    # --- WHO deaths: super-categoría icd10_group → causa sintética ---
    if spark.catalog.tableExists("stage.who_deaths_by_age_group_gtm"):
        cat_code = F.create_map(
            *[x for k, (c, n) in WHO_ICD10_CAT.items() for x in (F.lit(k), F.lit(c))]
        )
        cat_name = F.create_map(
            *[x for k, (c, n) in WHO_ICD10_CAT.items() for x in (F.lit(k), F.lit(n))]
        )
        dfs.append(
            spark.read.table("stage.who_deaths_by_age_group_gtm").select(
                F.lit(None).cast(StringType()).alias("cie10_code"),
                F.lit(None).cast(StringType()).alias("cie10_nombre"),
                cat_code[F.col("icd10_group")].alias("gbd_code"),
                cat_name[F.col("icd10_group")].alias("gbd_nombre"),
            )
        )

    if not dfs:
        print("[WARN] dim_causa: ninguna tabla stage encontrada; se crea vacía.")
        return spark.createDataFrame(
            [], "id_causa INT, `cie-10_code` STRING, `cie-10_nombre` STRING, gbd_code STRING, gbd_nombre STRING"
        )

    union = dfs[0]
    for df in dfs[1:]:
        union = union.unionByName(df, allowMissingColumns=True)

    # Normalizar 'null' literal y limpiar.
    union = union.select(
        F.when(F.col("cie10_code") == "null", F.lit(None)).otherwise(F.col("cie10_code")).alias("cie10_code"),
        F.col("cie10_nombre"),
        F.when(F.col("gbd_code") == "null", F.lit(None)).otherwise(F.col("gbd_code")).alias("gbd_code"),
        F.col("gbd_nombre"),
    )

    # Clave natural: CIE-10 individual si existe, si no GBD.
    union = union.withColumn("causa_nk", F.coalesce(F.col("cie10_code"), F.col("gbd_code")))
    union = union.filter(F.col("causa_nk").isNotNull())

    # Una fila por causa_nk: preferir la que trae CIE-10 real y nombres no nulos.
    w = Window.partitionBy("causa_nk").orderBy(
        F.col("cie10_code").isNull().asc(),
        F.col("cie10_nombre").isNull().asc(),
        F.col("gbd_nombre").isNull().asc(),
    )
    dim = (
        union.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1).drop("_rn")
    )
    dim = _assign_id(dim, "id_causa", "causa_nk")

    # ── Limpieza Kimball: NULL → genérico ──
    dim = dim.select(
        "id_causa",
        F.coalesce(F.col("cie10_code"),   F.lit("N/A")).alias("cie-10_code"),
        F.coalesce(F.col("cie10_nombre"), F.lit("No aplica")).alias("cie-10_nombre"),
        F.coalesce(F.col("gbd_code"),     F.lit("N/A")).alias("gbd_code"),
        F.coalesce(F.col("gbd_nombre"),   F.lit("No aplica")).alias("gbd_nombre"),
    )

    # ── Fila genérica (id=-1) para causas no mapeadas ──
    generic_row = spark.createDataFrame(
        [(GEN_ID_NEG, "N/A", "No aplica", "N/A", "No aplica")],
        ["id_causa", "cie-10_code", "cie-10_nombre", "gbd_code", "gbd_nombre"],
    )
    return generic_row.unionByName(dim)


# ═══════════════════════════════════════════════════════════════════════
# DIM_GEOGRAFIA
# ═══════════════════════════════════════════════════════════════════════

def build_dim_geografia() -> DataFrame:
    dfs = []

    if spark.catalog.tableExists("stage.ihme"):
        dfs.append(
            spark.read.table("stage.ihme").select(
                F.col("pais_iso3"), F.col("pais").alias("pais_nombre"),
                F.lit(None).cast(StringType()).alias("dep_ocurrencia"),
                F.lit(None).cast(StringType()).alias("mun_ocurrencia"),
            )
        )

    if spark.catalog.tableExists("stage.ine"):
        dfs.append(
            spark.read.table("stage.ine").select(
                F.col("pais_iso3"), F.lit("Guatemala").alias("pais_nombre"),
                F.col("dep_ocurrencia"), F.col("mun_ocurrencia"),
            )
        )

    if spark.catalog.tableExists("stage.mspas_mortalidad_general"):
        dfs.append(
            spark.read.table("stage.mspas_mortalidad_general").select(
                F.col("pais_iso3"), F.lit("Guatemala").alias("pais_nombre"),
                F.lit(None).cast(StringType()).alias("dep_ocurrencia"),
                F.lit(None).cast(StringType()).alias("mun_ocurrencia"),
            )
        )

    if spark.catalog.tableExists("stage.panama"):
        pan_stage = spark.read.table("stage.panama")
        pais_col = "pais" if "pais" in pan_stage.columns else "pais_iso3"
        dfs.append(
            pan_stage.select(
                F.col("pais_iso3"),
                F.when(F.col(pais_col).isNotNull(), F.col(pais_col))
                 .otherwise(F.lit("Panamá")).alias("pais_nombre"),
                F.lit(None).cast(StringType()).alias("dep_ocurrencia"),
                F.lit(None).cast(StringType()).alias("mun_ocurrencia"),
            )
        )

    for who_table in ("stage.who_deaths_by_age_group_gtm",
                       "stage.who_population_distribution_gtm"):
        if spark.catalog.tableExists(who_table):
            dfs.append(
                spark.read.table(who_table).select(
                    F.col("country_code").alias("pais_iso3"),
                    F.col("country_name").alias("pais_nombre"),
                    F.lit(None).cast(StringType()).alias("dep_ocurrencia"),
                    F.lit(None).cast(StringType()).alias("mun_ocurrencia"),
                )
            )
            break

    if not dfs:
        print("[WARN] dim_geografia: ninguna tabla stage encontrada.")
        return spark.createDataFrame(
            [], "id_geografia INT, pais_iso3 STRING, pais_nombre STRING, "
                "dep_ocurrencia STRING, mun_ocurrencia STRING"
        )

    union = dfs[0]
    for df in dfs[1:]:
        union = union.unionByName(df, allowMissingColumns=True)

    # ── Limpieza Kimball: NULL → "No aplica" ──
    dim = union.select(
        F.col("pais_iso3"),
        F.coalesce(F.col("pais_nombre"),     F.lit("No aplica")).alias("pais_nombre"),
        F.coalesce(F.col("dep_ocurrencia"),  F.lit("No aplica")).alias("dep_ocurrencia"),
        F.coalesce(F.col("mun_ocurrencia"),  F.lit("No aplica")).alias("mun_ocurrencia"),
    ).dropDuplicates(["pais_iso3", "dep_ocurrencia", "mun_ocurrencia"])

    dim = _assign_id(dim, "id_geografia", "pais_iso3", "dep_ocurrencia", "mun_ocurrencia")
    dim = dim.select("id_geografia", "pais_iso3", "pais_nombre",
                     "dep_ocurrencia", "mun_ocurrencia")

    generic_row = spark.createDataFrame(
        [(GEN_ID_NEG, "N/A", "No aplica", "No aplica", "No aplica")],
        ["id_geografia", "pais_iso3", "pais_nombre", "dep_ocurrencia",
         "mun_ocurrencia"],
    )
    return generic_row.unionByName(dim)


# ═══════════════════════════════════════════════════════════════════════
# DIM_TIEMPO  — DDL: anio_ocurrencia (no anio) + semana_epidemiologica
# Sin grano diario en el modelo → semana_epidemiologica = NULL.
# ═══════════════════════════════════════════════════════════════════════

def _build_tiempo_from_source(df: DataFrame, anio_col: str,
                               mes_col: str = None) -> DataFrame:
    cols = [F.col(anio_col).cast(IntegerType()).alias("anio_ocurrencia")]
    if mes_col:
        cols.append(F.col(mes_col).alias("mes_ocurrencia"))
    else:
        cols.append(F.lit("No aplica").alias("mes_ocurrencia"))
    return df.select(*cols).distinct()


def build_dim_tiempo() -> DataFrame:
    dfs = []

    if spark.catalog.tableExists("stage.ine"):
        dfs.append(
            _build_tiempo_from_source(
                spark.read.table("stage.ine"),
                "anio_ocurrencia", "mes_ocurrencia",
            )
        )

    for src, col in [
        ("stage.ihme",                          "anio"),
        ("stage.mspas_mortalidad_general",      "anio"),
        ("stage.mspas_tasa",                    "anio"),
        ("stage.panama",                        "anio"),
    ]:
        if spark.catalog.tableExists(src):
            dfs.append(_build_tiempo_from_source(spark.read.table(src), col))

    for who_table in ("stage.who_deaths_by_age_group_gtm",
                       "stage.who_population_distribution_gtm"):
        if spark.catalog.tableExists(who_table):
            dfs.append(
                _build_tiempo_from_source(spark.read.table(who_table), "year")
            )
            break

    if not dfs:
        print("[WARN] dim_tiempo: ninguna tabla stage encontrada.")
        return spark.createDataFrame(
            [], "id_tiempo INT, mes_ocurrencia STRING, mes_ocurrencia_num INT, "
                "anio_ocurrencia INT, es_pre_covid BOOLEAN"
        )

    union = dfs[0]
    for df in dfs[1:]:
        union = union.unionByName(df, allowMissingColumns=True)

    dim = union.dropDuplicates(["anio_ocurrencia", "mes_ocurrencia"])

    _months_df = spark.createDataFrame(
        [("Enero", 1), ("Febrero", 2), ("Marzo", 3), ("Abril", 4),
         ("Mayo", 5), ("Junio", 6), ("Julio", 7), ("Agosto", 8),
         ("Septiembre", 9), ("Octubre", 10), ("Noviembre", 11), ("Diciembre", 12)],
        ["mes", "num"]
    )
    dim = dim.join(F.broadcast(_months_df),
                   dim.mes_ocurrencia == _months_df.mes, "left") \
            .drop("mes")
    dim = dim.withColumn(
        "mes_ocurrencia_num",
        F.when(F.col("mes_ocurrencia") == "No aplica", F.lit(GEN_ID_ZERO))
         .when(F.col("mes_ocurrencia").isNull(),       F.lit(GEN_ID_ZERO))
         .otherwise(F.col("num"))
    ).drop("num")
    dim = dim.withColumn(
        "es_pre_covid",
        F.when(F.col("anio_ocurrencia").isNull(), F.lit(False))
         .when(F.col("anio_ocurrencia") <= 2019, F.lit(True))
         .otherwise(F.lit(False))
    )
    dim = dim.withColumn("id_tiempo", F.monotonically_increasing_id().cast(IntegerType()))
    dim = dim.select("id_tiempo", "mes_ocurrencia", "mes_ocurrencia_num",
                     "anio_ocurrencia", "es_pre_covid")

    generic_row = spark.createDataFrame(
        [(GEN_ID_NEG, "No aplica", GEN_ID_ZERO, -1, False)],
        "id_tiempo INT, mes_ocurrencia STRING, mes_ocurrencia_num INT,"
        " anio_ocurrencia INT, es_pre_covid BOOLEAN"
    )
    return generic_row.unionByName(dim)


# ═══════════════════════════════════════════════════════════════════════
# DIM_IHME_PERFIL  — DDL: id_ihme_perfil, metrica, medida
# ═══════════════════════════════════════════════════════════════════════

def build_dim_ihme_perfil() -> DataFrame:
    if not spark.catalog.tableExists("stage.ihme"):
        print("[WARN] dim_ihme_perfil: stage.ihme no existe.")
        return spark.createDataFrame(
            [], "id_ihme_perfil INT, metrica STRING, medida STRING"
        )

    dim = (
        spark.read.table("stage.ihme")
        .select("medida", "metrica")
        .distinct()
    )
    dim = _assign_id(dim, "id_ihme_perfil", "medida", "metrica")
    return dim.select("id_ihme_perfil", "metrica", "medida")


# ═══════════════════════════════════════════════════════════════════════
# DIM_INE_PERFIL  — canonicaliza categorías para no fragmentar el perfil.
# ═══════════════════════════════════════════════════════════════════════

def canon_ine_perfil(df: DataFrame) -> DataFrame:
    """Unifica variantes equivalentes de INE:
       'Casado(a)'->'Casado', 'Básico'->'Básica', 'Ninguno'->'Ninguna'.
       Se usa idéntico en facts.py para que el join calce."""
    # estado_civil: quitar sufijo "(a)"
    ec = F.trim(F.regexp_replace(F.col("estado_civil"), r"\(a\)\s*$", ""))
    # escolaridad: unificar género gramatical
    esc = F.when(F.col("escolaridad") == "Básico",  F.lit("Básica")) \
           .when(F.col("escolaridad") == "Ninguno", F.lit("Ninguna")) \
           .otherwise(F.col("escolaridad"))
    return df.withColumn("estado_civil", ec).withColumn("escolaridad", esc)


def build_dim_ine_perfil() -> DataFrame:
    if not spark.catalog.tableExists("stage.ine"):
        print("[WARN] dim_ine_perfil: stage.ine no existe.")
        return spark.createDataFrame(
            [], "id_ine_perfil INT, estado_civil STRING, escolaridad STRING, "
                "asistencia_medica STRING, tipo_ocurrencia STRING"
        )

    base = spark.read.table("stage.ine").select(
        "estado_civil", "escolaridad", "asistencia_medica", "tipo_ocurrencia"
    )
    base = canon_ine_perfil(base).distinct()

    # ── Limpieza Kimball: NULL → "Desconocido" ──
    dim = base.select(
        F.coalesce(F.col("estado_civil"),      F.lit("Desconocido")).alias("estado_civil"),
        F.coalesce(F.col("escolaridad"),       F.lit("Desconocido")).alias("escolaridad"),
        F.coalesce(F.col("asistencia_medica"), F.lit("Desconocido")).alias("asistencia_medica"),
        F.coalesce(F.col("tipo_ocurrencia"),   F.lit("Desconocido")).alias("tipo_ocurrencia"),
    ).distinct()

    dim = _assign_id(dim, "id_ine_perfil", "estado_civil", "escolaridad",
                     "asistencia_medica", "tipo_ocurrencia")

    generic_row = spark.createDataFrame(
        [(GEN_ID_NEG, "Desconocido", "Desconocido", "Desconocido", "Desconocido")],
        ["id_ine_perfil", "estado_civil", "escolaridad", "asistencia_medica", "tipo_ocurrencia"],
    )
    dim = generic_row.unionByName(dim)

    return dim.select("id_ine_perfil", "estado_civil", "escolaridad",
                      "asistencia_medica", "tipo_ocurrencia")


# ═══════════════════════════════════════════════════════════════════════
# EJECUCIÓN
# ═══════════════════════════════════════════════════════════════════════

def main():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DW}")

    print("=== Poblando dimensiones ===")

    _save(build_dim_etario(),       "dim_etario")
    _save(build_dim_source(),       "dim_source")
    _save(build_dim_genero(),       "dim_genero")
    _save(build_dim_causa(),        "dim_causa")
    _save(build_dim_geografia(),    "dim_geografia")
    _save(build_dim_tiempo(),       "dim_tiempo")
    _save(build_dim_ihme_perfil(),  "dim_ihme_perfil")
    _save(build_dim_ine_perfil(),   "dim_ine_perfil")

    print("=== Dimensiones pobladas ===")
    for d in ["dim_etario", "dim_source", "dim_genero", "dim_causa",
              "dim_geografia", "dim_tiempo", "dim_ihme_perfil", "dim_ine_perfil"]:
        display(spark.read.table(f"{SCHEMA_DW}.{d}").limit(20))


if __name__ == "__main__":
    main()
else:
    main()
